<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session03-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 3: in-class discussion problem (3 of 3)

This is the block 3 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then fix and run the code below, discuss the outcome with your neighbour, and answer the quiz again.

## Is NOT symmetric?

A PubMed search for X gives 50,000 articles, Y gives 30,000, and `X AND Y` gives 8,000.
`X NOT Y` gives 42,000. How many does `Y NOT X` give?

**Fix and run.** The first cell predicts the counts from the three numbers with set arithmetic, but
the line marked `FIX` has a copy-paste mistake. Run it once as it is: what does it predict for
`Y NOT X`? Then fix the line and run it again. The second cell checks the same arithmetic on real
PubMed counts, for CRISPR and cancer.

In [1]:
x, y, x_and_y = 50_000, 30_000, 8_000

x_or_y = x + y - x_and_y
x_not_y = x - x_and_y
y_not_x = x - x_and_y          # FIX

print(f"X OR Y    {x_or_y:>7,}")
print(f"X NOT Y   {x_not_y:>7,}")
print(f"Y NOT X   {y_not_x:>7,}")

X OR Y     72,000
X NOT Y    42,000
Y NOT X    42,000


In [2]:
# The same arithmetic on real PubMed counts (needs the internet)
import time
import requests

def pubmed_count(term):
    r = requests.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                     params={"db": "pubmed", "term": term, "retmode": "json"}, timeout=30)
    r.raise_for_status()
    time.sleep(0.4)            # NCBI allows three requests a second without an API key
    return int(r.json()["esearchresult"]["count"])

a, b, ab = pubmed_count("CRISPR"), pubmed_count("cancer"), pubmed_count("CRISPR AND cancer")
for term, predicted in [("CRISPR OR cancer", a + b - ab), ("CRISPR NOT cancer", a - ab),
                        ("cancer NOT CRISPR", b - ab)]:
    print(f"{term:18s} PubMed {pubmed_count(term):>10,}   predicted {predicted:>10,}")

CRISPR OR cancer   PubMed  5,773,974   predicted  5,773,974


CRISPR NOT cancer  PubMed     53,375   predicted     53,375


cancer NOT CRISPR  PubMed  5,702,287   predicted  5,702,287


**If you finish early.**

1. Without running anything: what is `X OR Y` when `X AND Y` gives 0? And when Y is completely
   inside X?
2. Search PubMed for `haemoglobin` and for `hemoglobin`. Does PubMed treat the two spellings as the
   same word? What does that mean for a `NOT` search?

**Discussion point.** As it was, the cell predicted 42,000 for `Y NOT X`, the same as
`X NOT Y`: the line was copied from the one above and still started from X. The fix starts from Y,

```python
y_not_x = y - x_and_y
```

and gives 22,000. `NOT` is not symmetric: `Y NOT X` starts from all of Y and removes the overlap, so
its size is Y minus the shared part. X and Y have different sizes, so removing the same 8,000 leaves
different remainders. `OR` is symmetric, and it counts the overlap only once: 50,000 + 30,000 − 8,000
= 72,000, not 80,000.

The real PubMed counts follow the same arithmetic, because Boolean operators are set operations on the
lists of matching articles. That is also why `NOT` is dangerous: it removes every article that
mentions the other term even once, including ones you wanted.